# Digital DNA — Music Metadata API Exploration

This notebook documents exploratory development for the MusicBrainz metadata portion of Digital DNA.

The goal is to understand the API response structure, transform candidate recordings into an analysis-friendly format, test matching rules, and validate the reusable ingestion workflow before implementing the production-style pipeline in Databricks.

Saved API response samples are used for repeatable analysis because the public MusicBrainz API can be temporarily unavailable.


## 1. Environment Setup

Add the project root to the Python path so the reusable modules in `src/` can be imported from this notebook.


In [3]:
import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

sample_dir = project_root / "data" / "sample"

print(f"Project root: {project_root}")


Project root: /Users/ashlyngrace/Documents/Projects/digital-dna


## 2. Inspect a Saved MusicBrainz Response

Start with the saved Espresso search response. This preserves the raw API structure while avoiding dependence on a live request during notebook execution.


In [5]:
espresso_sample_path = sample_dir / "musicbrainz_espresso_sample.json"

with open(espresso_sample_path, "r", encoding="utf-8") as file:
    espresso_data = json.load(file)

print("Top-level keys:", list(espresso_data.keys()))
print("Total MusicBrainz matches:", espresso_data.get("count"))
print("Candidates returned:", len(espresso_data.get("recordings", [])))


Top-level keys: ['created', 'count', 'offset', 'recordings']
Total MusicBrainz matches: 32
Candidates returned: 20


In [6]:
first_recording = espresso_data["recordings"][0]

{
    "recording_id": first_recording.get("id"),
    "title": first_recording.get("title"),
    "score": first_recording.get("score"),
    "first_release_date": first_recording.get("first-release-date"),
    "disambiguation": first_recording.get("disambiguation")
}


{'recording_id': '4c567421-9895-4ee0-a442-9c63cab23e07',
 'title': 'Espresso',
 'score': 100,
 'first_release_date': '2025-02-21',
 'disambiguation': 'live, 2024-12-20: NPR Music Office, Washington, D.C., USA'}

## 3. Transform Candidate Recordings

Use the reusable transformation module rather than rebuilding the parsing logic inside the notebook.


In [8]:
from src.transformation.recording_transform import recordings_to_dataframe

espresso_df = recordings_to_dataframe(espresso_data)

print("Candidate rows:", len(espresso_df))
espresso_df.head()


Candidate rows: 20


,recording_id,title,artist,score,first_release_date,disambiguation
0,4c567421-9895-4ee0-a442-9c63cab23e07,Espresso,Sabrina Carpenter,100,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington..."
1,cea42ffa-6d93-406c-964c-c4eb47c0a18b,Espresso,Sabrina Carpenter,100,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix
2,9867e32e-89ff-4b0c-a101-7d9126e114cd,Espresso,Sabrina Carpenter,100,2024-05-03,clean
3,a59823d1-3570-40eb-9317-227ded561779,Espresso,Sabrina Carpenter,100,None,"live, 2024-05-18: Saturday Night Live"
4,b013776b-4703-4f41-8743-25ac90abc623,Espresso,Sabrina Carpenter,100,2024-04-11,"Dolby Atmos mix, explicit"


In [9]:
espresso_df[
    [
        "recording_id",
        "title",
        "artist",
        "score",
        "first_release_date",
        "disambiguation"
    ]
]


,recording_id,title,artist,score,first_release_date,disambiguation
0,4c567421-9895-4ee0-a442-9c63cab23e07,Espresso,Sabrina Carpenter,100,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington..."
1,cea42ffa-6d93-406c-964c-c4eb47c0a18b,Espresso,Sabrina Carpenter,100,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix
2,9867e32e-89ff-4b0c-a101-7d9126e114cd,Espresso,Sabrina Carpenter,100,2024-05-03,clean
3,a59823d1-3570-40eb-9317-227ded561779,Espresso,Sabrina Carpenter,100,None,"live, 2024-05-18: Saturday Night Live"
4,b013776b-4703-4f41-8743-25ac90abc623,Espresso,Sabrina Carpenter,100,2024-04-11,"Dolby Atmos mix, explicit"
5,05e5ca93-eb60-491b-9669-8bac0307d52d,Espresso,Sabrina Carpenter,100,2024-12-20,part of “2024 MegaMix by Paris Hilton” DJ‐mix
6,d1b37355-1913-470b-87e7-4ac58c55483d,Espresso,Sabrina Carpenter,100,None,lyric video
7,d3efc8d3-90e7-4696-8eda-2cab5ebc050d,Espresso,Sabrina Carpenter,100,2024-04-11,explicit
8,c733559e-0294-40ca-b6a8-b4b022fdef9e,Espresso,Sabrina Carpenter,99,2024-04-13,"live, 2024‐04‐12: Coachella Stage, Indio, CA, USA"
9,80bb466a-f303-486e-adf0-b0d9968b5c32,Espresso,Sabrina Carpenter,99,2024-05-03,"Dolby Atmos mix, clean"


## 4. Validate and Rank Candidate Matches

MusicBrainz can return live versions, DJ mixes, commentary, and other variants. The validation layer assigns a match priority so preferred candidates can be separated from weak or rejected matches.


In [11]:
from src.validation.recording_validation import (
    get_match_priority,
    get_rejection_reason,
    is_preferred_version,
    select_best_match
)

validated_df = espresso_df.copy()

validated_df["is_preferred_version"] = validated_df["disambiguation"].apply(
    is_preferred_version
)

validated_df["rejection_reason"] = validated_df["disambiguation"].apply(
    get_rejection_reason
)

validated_df["match_priority"] = validated_df.apply(
    lambda row: get_match_priority(
        row["title"],
        "Espresso",
        row["disambiguation"]
    ),
    axis=1
)

validated_df = validated_df.sort_values(
    by=["match_priority", "score"],
    ascending=[False, False]
)

validated_df[
    [
        "title",
        "artist",
        "score",
        "first_release_date",
        "disambiguation",
        "rejection_reason",
        "match_priority"
    ]
]


,title,artist,score,first_release_date,disambiguation,rejection_reason,match_priority
11,Espresso,Sabrina Carpenter,99,2024-04-12,None,None,3
2,Espresso,Sabrina Carpenter,100,2024-05-03,clean,None,2
4,Espresso,Sabrina Carpenter,100,2024-04-11,"Dolby Atmos mix, explicit",None,2
7,Espresso,Sabrina Carpenter,100,2024-04-11,explicit,None,2
9,Espresso,Sabrina Carpenter,99,2024-05-03,"Dolby Atmos mix, clean",None,2
12,Espresso (Espressooooo Version),Sabrina Carpenter,89,2025-11-28,None,None,1
13,Espresso (espressooooo version),Sabrina Carpenter,89,2024-05-17,None,None,1
14,Espresso (on vacation),Sabrina Carpenter,89,2024-05-17,None,None,1
15,Espresso (espressooooo version),Sabrina Carpenter,89,2024-05-17,Dolby Atmos mix,None,1
16,Espresso (decaf version),Sabrina Carpenter,89,2024-05-17,None,None,1


In [12]:
print(
    validated_df["match_priority"]
    .value_counts()
    .sort_index()
)

best_match = select_best_match(
    espresso_df,
    "Espresso"
)

best_match


match_priority
0    7
1    8
2    4
3    1
Name: count, dtype: int64


recording_id          48ee49e2-b4db-47fd-96de-5ec2ac542a48
title                                             Espresso
artist                                   Sabrina Carpenter
score                                                   99
first_release_date                              2024-04-12
disambiguation                                        None
match_priority                                           3
Name: 11, dtype: object

## 5. Test the End-to-End Matching Function

The transformation module combines candidate parsing and best-match selection into one reusable function.


In [14]:
from src.transformation.recording_transform import process_recording_matches

best_match_from_sample = process_recording_matches(
    espresso_data,
    "Espresso"
)

best_match_from_sample


recording_id          48ee49e2-b4db-47fd-96de-5ec2ac542a48
title                                             Espresso
artist                                   Sabrina Carpenter
score                                                   99
first_release_date                              2024-04-12
disambiguation                                        None
match_priority                                           3
Name: 11, dtype: object

## 6. Inspect the Multi-Track Ingestion Structure

The ingestion workflow was expanded from a single search to multiple requested tracks. Each API response now travels with its own `requested_title` and `requested_artist`, allowing downstream processing to remain dynamic instead of hard-coded to one song.


In [16]:
multi_track_path = sample_dir / "musicbrainz_multi_track_sample.json"

with open(multi_track_path, "r", encoding="utf-8") as file:
    multi_track_data = json.load(file)

print("Tracks in sample:", len(multi_track_data))

for result in multi_track_data:
    print(
        result["requested_title"],
        "-",
        result["requested_artist"],
        ":",
        len(result["api_response"].get("recordings", [])),
        "candidates"
    )


Tracks in sample: 2
Espresso - Sabrina Carpenter : 20 candidates
Good Luck, Babe! - Chappell Roan : 12 candidates


In [17]:
multi_track_summary = pd.DataFrame(
    [
        {
            "requested_title": result["requested_title"],
            "requested_artist": result["requested_artist"],
            "total_matches": result["api_response"].get("count"),
            "candidates_returned": len(
                result["api_response"].get("recordings", [])
            )
        }
        for result in multi_track_data
    ]
)

multi_track_summary


,requested_title,requested_artist,total_matches,candidates_returned
0,Espresso,Sabrina Carpenter,32,20
1,"Good Luck, Babe!",Chappell Roan,12,12


## 7. Reusable Live Ingestion Function

The production ingestion module exposes `search_multiple_recordings()` for live MusicBrainz requests. It is imported here to verify that the reusable interface is available, but the notebook does not call the live API during `Run All` because transient 503 responses would make the exploratory notebook non-deterministic.


In [19]:
from src.ingestion.musicbrainz_api import (
    search_recording,
    search_multiple_recordings
)

print("Reusable ingestion functions loaded successfully.")


Reusable ingestion functions loaded successfully.


## Exploration Outcome

The exploration established a reusable metadata workflow:

**MusicBrainz response → candidate transformation → validation/ranking → multi-track request context → Databricks Bronze/Silver pipeline**

The Databricks notebooks contain the production-style medallion implementation. This notebook remains focused on API exploration, prototyping, and validation of the reusable Python modules.
